# Explicit input-gradient explanation

## Goal
Understand the saliency interface on synthetic inputs without claiming biological interpretation.

## Setup
Install the package in this kernel's environment. All default data are synthetic; research training is disabled.

In [ ]:
from pathlib import Path
import os, sys, uuid
import numpy as np
PROJECT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()), Path.cwd())
WORK = Path(os.environ.get("SERS_NOTEBOOK_OUTPUT", PROJECT / "output/notebooks")) / uuid.uuid4().hex[:8]
WORK.mkdir(parents=True)
from sersd2d_superraman.data import SpectrumBatch, load_batch, save_batch
from sersd2d_superraman.cli import synthetic, prepare

### 1. Explain a fixed logit target

In [ ]:
import torch
from sersd2d_superraman.classification.model import SuperRaman
from sersd2d_superraman.explainability import explain
torch.manual_seed(37)
model = SuperRaman().eval()
synthetic(WORK / "synthetic", per_class=5)
batch = load_batch(WORK / "synthetic/Enwave.npz")
gradient, classes = explain(model, batch.spectra[:1], target_classes=np.array([3]), target="logit")
assert gradient.shape == (1,1400) and np.isfinite(gradient).all()

### 2. Display the declared target and axis

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 1, figsize=(9,4.8), sharex=True, layout="constrained")
axes[0].plot(batch.spectra[0], color="#176b87")
axes[0].set_ylabel("Synthetic intensity")
axes[0].set_title("Untrained model: gradient of class-3 logit on a synthetic spectrum")
axes[1].plot(gradient[0], color="#9b3b54")
axes[1].set_ylabel("Logit input gradient")
axes[1].set_xlabel("Spectral bin index (physical Raman axis unavailable)")
fig.savefig(WORK / "saliency.png", dpi=150)
plt.show()

## Checks and next steps
Gradients are with respect to classifier inputs scaled to [-1,1]. For a verified model, choose logit or probability targets and record class IDs; SmoothGrad uses fixed targets and a recorded noise seed. Supply an actual Raman axis before showing cm^-1. SHAP and paper BCE-based explanations are separately deferred extensions.